# بايبلاين صور الأداء (Performers) — تحميل + فحص وجوه + رفع مشفّر على Hugging Face Bucket

النوتبوك ده بيشغّل البايبلاين المقسّم على **10 دفعات**، وكل دفعة بتعدي بالخطوات دي:

1. تحميل الصور الأصلية بتاعة الدفعة.
2. فحص وجود وجه فيها بموديل YOLOv11l-face وكتابة تقرير.
3. تحويل الصور **الأصلية** (مش نواتج الموديل) لملف Parquet واحد للدفعة.
4. تشفير ملف الـ Parquet بـ AES-256-GCM بكلمة سر واحدة لكل الدفعات العشرة (بمفتاح مختلف مشتق لكل دفعة).
5. رفع الملف المشفّر + تقرير فحص الوجوه بتاع الدفعة لـ Hugging Face Bucket.
6. التأكد من الرفع (verify) على السيرفر.
7. حذف صور الدفعة والملف المشفّر محلياً لتوفير المساحة، وبعدين الانتقال للدفعة التالية.

بعد كل دفعة بيتحدّث **تقرير رئيسي (master report)** ويترفع فوق نفسه على الـ Bucket — ده بيبقى نقطة استئناف: لو الجلسة اتقفلت لأي سبب وشغّلت النوتبوك تاني، البايبلاين هيقرأ آخر تقرير رئيسي من الـ Bucket ويكمل من الدفعة اللي بعدها بدل ما يعيد اللي خلص.

> ⚠️ لازم تفعّل **GPU** و **Internet** من (Settings) في النوتبوك قبل التشغيل.

## مساحة التخزين: `/kaggle/temp` مش `/kaggle/working`

حجم البيانات الكامل تقريباً 80 جيجا، وده أكبر من مساحة `/kaggle/working` (~20 جيجا أو أقل).
لذلك كل التحميل والملفات المؤقتة بتاعة الدفعات بتتحط في `/kaggle/temp` (~50 جيجا) وبتتمسح أول بأول بعد رفع كل دفعة، فمش محتاجين مساحة أكبر من دفعة واحدة في نفس الوقت.

## 1) الإعدادات والمفاتيح

املأ القيم دي قبل التشغيل. لو مسجّل Kaggle Secrets بنفس الأسماء (`HF_TOKEN`, `PARQUET_PASSWORD`) هيتقروا تلقائياً بدل ما تكتبهم هنا كنص صريح.

In [ ]:
# ==== إعدادات المشروع ====
REPO_URL = "https://github.com/kareemkamal10/app.git"

# رابط ملف الـ JSON بتاع الأداء (المصدر)
JSON_SOURCE = "https://huggingface.co/datasets/abdelwahabnabil500/datafile/resolve/main/stashdb_performers_full.json"

# اسم الـ Hugging Face Bucket اللي هيترفع عليه كل حاجة
BUCKET = "abdelwahabnabil500/faces"

# عدد الدفعات، عدد الـ workers للتحميل المتوازي، وحجم الـ batch بتاع موديل الوجوه
BATCH_COUNT = 10
WORKERS = 16
DETECT_BATCH_SIZE = 32
DEVICE = "0"  # رقم الـ GPU

# ==== المفاتيح السرية ====
# متفضّلش تكتبهم هنا كنص صريح — الأفضل تسجّلهم في Kaggle Secrets
# (Add-ons -> Secrets) بنفس الاسمين: HF_TOKEN و PARQUET_PASSWORD
HF_TOKEN = ""
PARQUET_PASSWORD = ""

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ("HF_TOKEN", "PARQUET_PASSWORD"):
        try:
            _val = _secrets.get_secret(_name)
        except Exception:
            _val = None
        if _val:
            globals()[_name] = _val
            print(f"تم قراءة {_name} من Kaggle Secrets.")
except Exception:
    pass  # Kaggle Secrets مش متاحة، هنستخدم القيم المكتوبة فوق

assert REPO_URL, "REPO_URL لازم يكون متملي."
assert BUCKET, "BUCKET لازم يكون متملي."
assert JSON_SOURCE, "JSON_SOURCE لازم يكون متملي."
assert HF_TOKEN, "HF_TOKEN فاضي — سجّله في Kaggle Secrets أو اكتبه فوق."
assert PARQUET_PASSWORD, "PARQUET_PASSWORD فاضي — سجّله في Kaggle Secrets أو اكتبه فوق."


## 2) فحص الـ GPU

In [ ]:
!nvidia-smi

## 3) استنساخ المشروع من GitHub

In [ ]:
import os, shutil

PROJECT_DIR = "/kaggle/working/project"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

!git clone "{REPO_URL}" "{PROJECT_DIR}"


In [ ]:
%cd {PROJECT_DIR}
!ls -la

## 4) تثبيت المتطلبات

In [ ]:
!pip install -q -r requirements.txt

## 5) تشغيل البايبلاين الكامل (10 دفعات)

الخلية دي بتشغّل `batch_pipeline.py` بمساره الكامل، وبتستخدم `/kaggle/temp` كمساحة عمل مؤقتة.
لو الجلسة اتقطعت وشغّلتها تاني، هتقرأ آخر تقرير رئيسي من الـ Bucket وتكمل من الدفعة اللي بعدها تلقائياً.

In [ ]:
import os, subprocess, sys

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["PARQUET_PASSWORD"] = PARQUET_PASSWORD

WORK_DIR = "/kaggle/temp/pipeline_work"  # مش /kaggle/working -- عشان المساحة (50 جيجا بدل 20)

cmd = [
    sys.executable, "batch_pipeline.py",
    "--json-source", JSON_SOURCE,
    "--bucket", BUCKET,
    "--batch-count", str(BATCH_COUNT),
    "--workers", str(WORKERS),
    "--device", DEVICE,
    "--batch-size", str(DETECT_BATCH_SIZE),
    "--work-dir", WORK_DIR,
]

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"البايبلاين فشل بكود خروج {process.returncode} -- شغّل الخلية تاني وهيكمل من آخر دفعة اترفعت.")
print("\nخلص البايبلاين بنجاح.")


## 6) عرض التقرير الرئيسي النهائي

In [ ]:
import json
from pathlib import Path

master_path = Path(WORK_DIR) / "reports" / "download_master_report.md"
if master_path.exists():
    print(master_path.read_text(encoding="utf-8"))
else:
    print("مفيش تقرير رئيسي محلي -- شوف نسخته على الـ Bucket في final/download_master_report.md")
